In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Function to calculate the partial ozone column between given pressure levels (in hPa)
def calc_parc_o3(var, p_top=30, p_bottom=70):
    """
    Calculate the partial ozone column between specified pressure levels and convert to Dobson Units (DU).

    Parameters:
        var (xarray.DataArray): Ozone data with a pressure level coordinate.
        p_top (float): Top pressure level in hPa.
        p_bottom (float): Bottom pressure level in hPa.

    Returns:
        xarray.DataArray: Partial ozone column (DU) with zeros removed.
    """
    # Mean molecular mass of air per molecule (kg)
    m_air = 28.964 / (6.022e23)
    m_o3 = 47.998 / (6.022e23)
    var = var / m_o3 * m_air
    g = 980.6  # gravitational acceleration (cm/s^2)

    # Determine the pressure level coordinate name
    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    elif 'level' in var.dims:
        plev = var.level
        level = 'level'
    else:
        raise ValueError("No pressure level coordinate found in the data.")

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # Determine conversion factors based on the ordering and units of the pressure levels
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100         # convert from hPa to Pa
        factor_2 = 1         # plev already in hPa
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1
        factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100

    # Calculate pressure differences along the vertical axis
    if plev[0] < plev[-1]:
        # Increasing pressure levels
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level],
                                 coords={'time': time, level: plev})
        O3_col = var * weights_p * 10 / (g * m_air)
        O3_col = O3_col.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3_col = O3_col.sum(dim=level) / 2.687e16  # Convert to DU
    else:
        # Decreasing pressure levels
        for i in range(0, len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level],
                                 coords={'time': time, level: plev})
        O3_col = var * weights_p * 10 / (g * m_air)
        O3_col = O3_col.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3_col = O3_col.sum(dim=level) / 2.687e16

    return O3_col.where(O3_col != 0)

# Function to perform spatial averaging over longitude and a cosine-weighted average over latitude (60°N to 90°N)
def spatial_average(ozone_data, lat_start=60, lat_end=90):
    """
    Compute the spatial average of the ozone data:
      1. Average over longitude.
      2. Cosine-weighted average over the specified latitude range.

    Parameters:
        ozone_data (xarray.DataArray): Input ozone data.
        lat_start (float): Starting latitude.
        lat_end (float): Ending latitude.

    Returns:
        xarray.DataArray: Spatially averaged ozone data.
    """
    # Average over the longitude dimension
    ozone_avg = ozone_data.mean(dim='lon')
    # Select the specified latitude range
    lat_slice = ozone_avg.sel(lat=slice(lat_start, lat_end))
    # Compute cosine weights for latitude
    weights = np.cos(np.deg2rad(lat_slice.lat))
    ozone_weighted = lat_slice.weighted(weights).mean(dim='lat')
    return ozone_weighted

# Define file path for the MERRA2 dataset and output directory
MERRA2_FILE = "/mnt/backup_ETH/Marina/MERRA2/daily_fields/MERRA2_3d_dm_r144x96_1980-05.2020_SLP_O3.nc"
output_dir = '/home/weiji/restart_exam/plots/MERRA2_O3/'
os.makedirs(output_dir, exist_ok=True)

# Open the MERRA2 dataset and extract the O3 variable
ds = xr.open_dataset(MERRA2_FILE)
o3 = ds['O3']

# Compute the partial ozone column (30-70 hPa) using the provided method
o3_column = calc_parc_o3(o3, p_top=30, p_bottom=70)

# Compute spatial average over the Arctic region (60°N–90°N)
o3_column_avg = spatial_average(o3_column, lat_start=60, lat_end=90)

# ---------------------------
# Plot 1: Overall Evolution Plot
# ---------------------------
plt.figure(figsize=(30, 3))
plt.plot(o3_column_avg.time, o3_column_avg, color='black', linewidth=2, label='O3 Column (30–70 hPa)')
plt.xlabel('Time', fontsize=12)
plt.ylabel('Partial Ozone Column (DU)', fontsize=12)
plt.title('Evolution of Partial Ozone Column (30–70 hPa) in the Arctic (60°N–90°N)', fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.margins(x=0)
overall_plot_path = os.path.join(output_dir, "MERRA2_O3_Column_Evolution.png")
plt.savefig(overall_plot_path, dpi=150)
plt.show()

# ---------------------------
# Plot 2: 2020 Data with Envelope Shading
# ---------------------------
# Extract 2020 data from the overall time series
o3_2020 = o3_column_avg.sel(time=o3_column_avg.time.dt.year == 2020)

# Compute the envelope (min and max) using all available years by grouping by day-of-year
grouped = o3_column_avg.groupby('time.dayofyear')
envelope_min = grouped.min()
envelope_max = grouped.max()

# Align the envelope to the 2020 time coordinate using day-of-year
# Create envelope arrays for 2020 by selecting the corresponding min and max for each day
doy_2020 = o3_2020.time.dt.dayofyear
env_min_2020 = xr.DataArray(
    [float(envelope_min.sel(dayofyear=int(d)).values) for d in doy_2020.values],
    coords=o3_2020.time.coords, dims="time"
)
env_max_2020 = xr.DataArray(
    [float(envelope_max.sel(dayofyear=int(d)).values) for d in doy_2020.values],
    coords=o3_2020.time.coords, dims="time"
)

# Create the 2020 plot with envelope shading
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

# Plot the envelope shading (light gray) using the min and max from all years
ax.fill_between(o3_2020.time, env_min_2020, env_max_2020, color='royalblue', alpha=0.3, label='History Min-Max')

# Plot the 2020 ozone column as a black line
ax.plot(o3_2020.time, o3_2020, color='black', linewidth=5, label='2020')

# Set x-axis to show months from Jan to Dec
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_xlabel('Month', fontsize=16)
ax.set_ylabel('Partial Ozone Column, 30–70 hPa (DU)', fontsize=18)
ax.tick_params(axis='y', labelsize=16)
ax.set_title('2020 Partial Ozone Column (30–70 hPa)', fontsize=18)

# Add a text label indicating the year in the upper left corner
ax.text(0.02, 0.95, 'Year: 2020', transform=ax.transAxes, fontsize=18,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.margins(x=0)
ax.legend(fontsize=14)

# Save and display the 2020 envelope plot
envelope_plot_path = os.path.join(output_dir, "MERRA2_O3_2020_Envelope.png")
fig.savefig(envelope_plot_path, dpi=150)
plt.show()
